# Classes, errors and tests

This notebook covers the first block of Session 2. Session 1 used functions briefly; here we start with them properly, because a class is built from functions. We then define classes and objects, combine classes (composition and inheritance), let dataclasses remove boilerplate, document a contract with type hints and docstrings, report broken contracts with exceptions, and check with automated tests (pytest) that the code keeps its promises. The running example is a class that validates and cleans one Airbnb listing of the course case study, `ListingRecord`, in the [workspace](../workspace/README.md). The raw file of Inside Airbnb stores the price as text (`"$1,083.00"`), and nothing stops a row from having coordinates outside Berlin or an unknown room type; a class that checks every listing once, when it is created, protects all code that uses it.

```mermaid
flowchart LR
  F["Functions<br/>(one rule, named)"] --> C["Classes<br/>(contract: fields, methods)"]
  C --> H["Type hints and<br/>docstrings<br/>(written contract)"]
  C --> E["Exceptions<br/>(contract broken)"]
  H --> T["Tests<br/>(contract checked)"]
  E --> T
  T --> R["Code others<br/>can rely on"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Functions: parameters, return values and side effects

A **function** is a named piece of code with **parameters** (the names in its definition) that receives **arguments** (the values in a call) and **returns** a value to the caller.

- A parameter can have a **default value**; the argument then becomes optional.
- Arguments can be passed by position (`stay_cost(75, 3)`) or by name, as **keyword arguments** (`stay_cost(75, nights=3)`). Keywords make calls readable and protect against swapping two numbers.
- The **docstring**, the string on the first line of the body, says what the function does; `help(stay_cost)` shows it.
- `return` ends the function and hands back one value (which may be a tuple of several). A function without `return` returns `None`.

A **pure function** computes its result only from its arguments and changes nothing else: same input, same output. A function with a **side effect** also changes something outside itself: it prints, writes a file, or modifies an argument such as a list. Side effects are sometimes the purpose (saving a file), but pure functions are much easier to test and to reuse.

In [2]:
def stay_cost(nightly_price: float, nights: int, cleaning_fee: float = 25) -> float:
    """Return the total cost of a stay; the cleaning fee is paid once."""
    return nightly_price * nights + cleaning_fee


print(stay_cost(75, 3))                       # 250: default fee
print(stay_cost(75, nights=3, cleaning_fee=0))  # 225: keyword arguments


def add_fee_in_place(prices: list[float], fee: float) -> None:
    """Side effect: changes the list it receives and returns None."""
    for i in range(len(prices)):
        prices[i] += fee


def with_fee(prices: list[float], fee: float) -> list[float]:
    """Pure: returns a new list and leaves its argument unchanged."""
    return [p + fee for p in prices]


prices = [75.0, 90.0]
print(with_fee(prices, 25), prices)          # [100.0, 115.0] [75.0, 90.0]
print(add_fee_in_place(prices, 25), prices)  # None [100.0, 115.0]

250
225
[100.0, 115.0] [75.0, 90.0]
None [100.0, 115.0]


## Classes and objects: attributes and methods

A **class** is a blueprint for a kind of thing, for example "a listing". An **object** (or **instance**) is one concrete thing made from that blueprint: the flat with id 3176 in Pankow. There is one class and there can be any number of objects, each with **its own data**.

- **Attributes** are the data of an object, accessed with a dot: `flat.price`.
- **Methods** are functions defined inside the class; they describe what the objects can do: `flat.price_per_guest()`.
- `__init__` is the method Python calls when an object is created with `Listing(...)`; it stores the arguments as attributes.
- `self` is the first parameter of every method: the object the method is called on.

**Function or method?** A method *is* a function, defined in a class and called on an object. When you write `flat.price_per_guest()`, Python calls the function `Listing.price_per_guest` and passes `flat` as `self`. You already used methods in Session 1: `"Pankow".upper()` is the same call as `str.upper("Pankow")`, and `prices.append(85)` calls the method `append` of one particular list.

```mermaid
classDiagram
  class Listing {
    +int id
    +str district
    +float price
    +int guests
    +price_per_guest() float
  }
```

In [3]:
class Listing:
    """One accommodation listing."""

    def __init__(self, id: int, district: str, price: float, guests: int) -> None:
        self.id = id                  # attributes: stored on this object
        self.district = district
        self.price = price
        self.guests = guests

    def price_per_guest(self) -> float:   # a method: self is the object it is called on
        return round(self.price / self.guests, 2)

    def __repr__(self) -> str:            # how the object is shown
        return f"Listing({self.id}, {self.district!r}, {self.price})"


flat = Listing(3176, "Pankow", 160.71, 2)   # one object (instance) of the class
loft = Listing(4242, "Mitte", 230.00, 4)    # another object, with its own data
print(flat, loft)
print(flat.price_per_guest(), loft.price_per_guest())
print(Listing.price_per_guest(flat))        # the same call, written as a function
print(type(flat).__name__, isinstance(flat, Listing))

Listing(3176, 'Pankow', 160.71) Listing(4242, 'Mitte', 230.0)
80.36 57.5
80.36
Listing True


`flat` and `loft` are two objects of one class. Changing `flat.price` does not change `loft`, because each object owns its attributes. A class is worth writing when several functions work on the same data: the data and the rules that belong to it then live in one place.

> 📖 **Read more:** [extras/01-classes-and-functions.ipynb](extras/01-classes-and-functions.ipynb) (Think Python, ch. 14): a `Time` class built step by step: attributes, objects as return values, mutability and copying, pure functions.
>
> 📖 **Read more:** [extras/02-classes-and-methods.ipynb](extras/02-classes-and-methods.ipynb) (Think Python, ch. 15): the same functions turned into methods, with `__init__`, `__str__` and operator overloading.

## Composition, inheritance and dataclasses

### Concept

Two ways of building larger classes from smaller ones:

- **Composition** ("has a"): an object holds other objects as attributes and delegates work to them. A `FeatureSet` *has* a list of features; the workspace's `ListingTable` *has* a pandas DataFrame.
- **Inheritance** ("is a"): a **subclass** takes over all attributes and methods of its **base class** (or parent) and adds or replaces some. `DigitCount` *is a* `Feature`. Replacing a method of the parent is called **overriding**. Code that works with `Feature` objects works with every subclass: this is **polymorphism**.

A **dataclass** is a class whose main purpose is to hold data. The decorator `@dataclass` reads the annotated class attributes (the **fields**) and writes `__init__`, `__repr__` and `__eq__` automatically. The special method `__post_init__` runs right after `__init__` and is the place to check and clean the fields. `frozen=True` makes the objects read-only.

A small example by hand: `FeatureSet([WordCount(), DigitCount()])` applied to the listing title `"Cosy flat, 45 m²"` asks each feature in turn: four words, two digits (the ² is not a digit), so the result is `{"n_words": 4, "n_digits": 2}`.

```mermaid
classDiagram
  class Feature {
    <<abstract>>
    +str name
    +compute(text) int
  }
  class WordCount {
    +compute(text) int
  }
  class DigitCount {
    +compute(text) int
  }
  class FeatureSet {
    +list~Feature~ features
    +transform(text) dict
  }
  Feature <|-- WordCount : inherits
  Feature <|-- DigitCount : inherits
  FeatureSet o-- Feature : has many
```

### Why it matters

Composition keeps classes small and replaceable: a `FeatureSet` can be given other features, or a simple fake one in a test, without changing its own code. Inheritance lets many classes share one interface, so that the code that uses them does not need to know which one it has. Dataclasses make data-holding classes short enough to read at a glance. Used carelessly, deep inheritance trees become hard to follow; a common rule is *prefer composition, inherit only for a true "is a" relation*.

### How it works in Python

In [4]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field


class Feature(ABC):
    """Base class: a feature turns one text into one number."""

    name = "feature"

    @abstractmethod
    def compute(self, text: str) -> int:   # every subclass must implement this
        ...


class WordCount(Feature):                  # inheritance: WordCount is a Feature
    name = "n_words"

    def compute(self, text: str) -> int:   # overrides the abstract method
        return len(text.split())


class DigitCount(Feature):
    name = "n_digits"

    def compute(self, text: str) -> int:
        return sum(c in "0123456789" for c in text)


@dataclass
class FeatureSet:                          # composition: a FeatureSet has features
    features: list[Feature] = field(default_factory=list)

    def transform(self, text: str) -> dict[str, int]:
        return {f.name: f.compute(text) for f in self.features}   # polymorphism


features = FeatureSet([WordCount(), DigitCount()])
print(features.transform("Cosy flat, 45 m²"))  # {'n_words': 4, 'n_digits': 2}
print(isinstance(WordCount(), Feature))        # True
try:
    Feature()                                  # an abstract class cannot be instantiated
except TypeError as err:
    print("TypeError:", str(err).split(" with")[0])   # TypeError: Can't instantiate abstract class Feature


@dataclass
class Listing:
    """A dataclass: __init__, __repr__ and __eq__ are generated from the fields."""

    id: int
    room_type: str
    name: str = ""

    def __post_init__(self) -> None:           # runs after the generated __init__
        self.name = " ".join(self.name.split())


flat = Listing(3176, "Entire home/apt", "  Fabulous   Flat ")
print(flat)                                    # Listing(id=3176, room_type='Entire home/apt', name='Fabulous Flat')
print(flat == Listing(3176, "Entire home/apt", "Fabulous Flat"))   # True: fields are compared

{'n_words': 4, 'n_digits': 2}
True
TypeError: Can't instantiate abstract class Feature
Listing(id=3176, room_type='Entire home/apt', name='Fabulous Flat')
True


### In practice

- scikit-learn builds every model on the base class `BaseEstimator` and small "mixin" classes (`ClassifierMixin`, `TransformerMixin`). Because all models share `fit` and `predict`, a `Pipeline` can combine any of them: composition and inheritance together.
- Python's `dataclasses` module (PEP 557, Python 3.7) is widely used for configuration and record types; the library pydantic, used by FastAPI, extends the same idea with automatic type conversion and validation.

> **Warning.** A mutable default such as `features: list = []` in a dataclass raises `ValueError` (in a normal function default it would be silently shared between all calls). Use `field(default_factory=list)`.

> **Tip.** Before you write a subclass, ask whether "B is an A" is true in the domain. A `ListingTable` *has* listings; it is not a kind of listing. When in doubt, use composition.

> 📖 **Read more:** [extras/03-classes-and-objects.ipynb](extras/03-classes-and-objects.ipynb) (Think Python, ch. 16): objects inside objects (composition), equivalence and identity, copying.
>
> 📖 **Read more:** [extras/04-inheritance.ipynb](extras/04-inheritance.ipynb) (Think Python, ch. 17): cards and decks, parent and child classes, specialisation.

## Type hints and docstrings

### Concept

A function or class is a **contract**: given inputs of a stated kind, it returns an output with a stated meaning. The contract is written down in two places.

- **Type hints** state the types of parameters, return values and fields: `def n_words(text: str) -> int`. Common forms: `list[str]`, `dict[str, int]`, `tuple[int, int]`, `str | None` (text or nothing), `Callable[[float], None]`. Python does **not** check them when the code runs. Editors use them for completion and warnings; **type checkers** such as mypy or ty check them without running the code.
- A **docstring** is the string literal on the first line of a module, class or function. It explains what the code does, its arguments, its result, the exceptions it raises and an example. `help(function)` and editors display it; documentation generators build web pages from it. The workspace uses the *Google style* (`Args:`, `Returns:`, `Raises:`).

### Why it matters

Hints and docstrings tell a teammate (and an AI assistant) how to call the code without reading its body. Type checkers catch a class of bugs before any test runs, for example passing a whole DataFrame where one title is expected. In code review, a missing or wrong docstring is as much a defect as a wrong line.

### How it works in Python

In [5]:
def share_with(texts: list[str], char: str = "!") -> float | None:
    """Share of texts that contain ``char``.

    Args:
        texts: listing titles.
        char: the character to look for.

    Returns:
        A share between 0 and 1, or None if ``texts`` is empty.
    """
    if not texts:
        return None
    return sum(char in t for t in texts) / len(texts)


print(share_with(["Best view!", "Quiet room"]))   # 0.5
print(share_with([]))                          # None
print(share_with.__annotations__["return"])    # float | None
print(share_with.__doc__.splitlines()[0])      # Share of texts that contain ``char``.
try:
    share_with(["Best view!"], char=5)         # the hint says str; Python does not stop the call
except TypeError as err:
    print("TypeError:", err)                   # TypeError: 'in <string>' requires string as left operand, not int

0.5
None
float | None
Share of texts that contain ``char``.
TypeError: 'in <string>' requires string as left operand, not int


Checking the hints statically, without running the code:

```bash
uvx ty check src/          # or: uvx mypy src/
```

### In practice

- pandas, NumPy and scikit-learn document every public function in the *NumPy docstring style*; their online API reference is generated from these docstrings.
- Dropbox added type hints to about four million lines of Python and checks them with mypy; the company described the migration in "Our journey to type checking 4 million lines of Python" (2019).

> **Note.** `from __future__ import annotations` at the top of a module (used in the workspace) stores hints as text and allows the newer syntax such as `str | None` on older Python versions.

> **Caution.** A hint is a promise, not a check. `price: float` does not stop someone from passing the text `"$85"` from the raw file; `price + 10` then raises a `TypeError`, and `price * 2` silently gives `"$85$85"`. Values that come from outside (files, forms, APIs) must be checked explicitly, which is the job of the validation in `__post_init__`.

## Exceptions and error handling

### Concept

An **exception** is Python's signal that an operation cannot be completed. Built-in examples: `ValueError` (right type, wrong value), `TypeError` (wrong type), `KeyError` (missing key), `FileNotFoundError`. If nothing handles an exception, the program stops and prints a **traceback**: the chain of calls that led to the error. Read it from the bottom: the last line names the exception and its message.

- `raise ValueError("message")` signals a broken contract deliberately.
- `try: … except ValueError as err: …` **handles** an expected exception; `else:` runs if no exception occurred; `finally:` always runs (for example to close a file).
- Exceptions are classes and form a hierarchy. Defining your own subclasses, such as `ValidationError(ListingToolsError, ValueError)`, lets callers catch exactly what they can handle.
- `raise NewError(...) from err` keeps the original cause in the traceback.

Two styles: *Look before you leap* (LBYL) checks first (`if key in d:`); *Easier to ask forgiveness than permission* (EAFP) tries and handles the exception. Python code often uses EAFP for conversions such as `int(value)`.

```mermaid
flowchart TD
  I["Raw value"] --> V{"Valid?"}
  V -- yes --> C["Clean and store"]
  V -- no --> R["raise ValidationError(field, message)"]
  R --> H{"Caller can<br/>handle it?"}
  H -- "yes (skip and count<br/>the record, report it)" --> L["Handle and log"]
  H -- no --> S["Stop with traceback"]
```

### Why it matters

A clear error at the place where the problem occurs saves hours compared with a wrong number found weeks later in a chart. Silent failures are worse than crashes because they produce plausible results. Catching every exception with a bare `except:` hides programming errors.

### How it works in Python

In [6]:
class ListingToolsError(Exception):
    """Base class of the package's errors."""


class ValidationError(ListingToolsError, ValueError):
    """A record breaks a rule; also a ValueError for callers that expect one."""

    def __init__(self, field: str, message: str) -> None:
        self.field = field
        super().__init__(f"{field}: {message}")


def parse_price(value: object) -> float:
    """Parse a price as written in the raw Inside Airbnb file: "$1,083.00"."""
    text = str(value).strip().lstrip("$").replace(",", "")
    try:
        price = float(text)                                         # EAFP
    except ValueError as err:
        raise ValidationError("price", f"expected a price such as $85.00, got {value!r}") from err
    if not price > 0:
        raise ValidationError("price", f"must be above 0, got {price}")
    return price


good, bad = [], []
for raw in ["$160.71", "85 EUR", "$1,083.00", "$0.00", "$97.33"]:
    try:
        price = parse_price(raw)
    except ValidationError as err:               # handle only the error we expect
        bad.append((err.field, str(err)))
    else:
        good.append(price)

print(good)        # [160.71, 1083.0, 97.33]
print(bad[0])      # ('price', "price: expected a price such as $85.00, got '85 EUR'")
print(bad[1])      # ('price', 'price: must be above 0, got 0.0')

[160.71, 1083.0, 97.33]
('price', "price: expected a price such as $85.00, got '85 EUR'")
('price', 'price: must be above 0, got 0.0')


### In practice

- In 2020, Public Health England failed to report about 16,000 positive COVID-19 test results because a data pipeline stored them in the old Excel format (`.xls`), whose limit of 65,536 rows silently cut the data. A pipeline that raised an error on truncation would have stopped instead of under-reporting.
- scikit-learn raises `ValueError: Input X contains NaN` instead of fitting a model on missing values; the message names the problem and the fix.

> **Warning.** Never write `except: pass` or `except Exception: pass`. It hides typing mistakes (`NameError`) together with the error you meant to catch. Catch specific exceptions and do something meaningful: skip and count the record, log it, or re-raise with a clearer message.

## Automated tests with pytest

### Concept

A **unit test** is a small function that calls the code with a known input and asserts the expected output. **pytest** finds files named `test_*.py`, runs every function whose name starts with `test_`, and reports each as passed, failed or skipped.

- `assert expression` fails the test if the expression is false; pytest shows the values involved.
- `@pytest.mark.parametrize("x, expected", [...])` runs one test for many cases, each reported separately.
- A **fixture** (`@pytest.fixture`) prepares data for tests; a test receives it by naming it as a parameter. Fixtures in `conftest.py` are available to all test files. `tmp_path` is a built-in fixture giving a fresh temporary folder.
- `with pytest.raises(ValidationError):` checks that the code raises the expected exception.
- `@pytest.mark.skip` skips a test (the workspace uses it for exercises).

Good tests are **small** (one behaviour each), **independent** (no shared state, any order), **fast** and **deterministic** (no network, no randomness without a seed). They cover normal cases, **edge cases** (empty text, a price with a thousands separator, a listing exactly on the city boundary) and invalid input.

```mermaid
stateDiagram-v2
  [*] --> Red: write a test that fails
  Red --> Green: write the simplest code that passes
  Green --> Refactor: clean up, tests stay green
  Refactor --> Red: next behaviour
```

This cycle is called **test-driven development** (TDD). You do not have to follow it strictly, but writing the test first makes you decide what "correct" means before you write the code.

### Why it matters

Tests turn "it worked when I tried it" into a check that runs on every change, on every computer and, with continuous integration (block 2), on every pull request. They make it safe to change shared code, and they document the expected behaviour by example.

### How it works in Python

Save as `test_price.py` and run `uv run --with pytest pytest -q test_price.py`, or run the file directly with Python (the last two lines start pytest):

`test_price.py`:

```python
import pytest


def price_to_number(text: str) -> float:
    if not isinstance(text, str):
        raise ValueError(f"expected a price as text, got {text!r}")
    try:
        return float(text.strip().lstrip("$").replace(",", ""))
    except ValueError:
        raise ValueError(f"expected a price as text, got {text!r}") from None


@pytest.fixture
def raw_prices() -> list[str]:
    return ["$160.71", "$1,083.00", "$49.00"]


def test_prices_of_fixture(raw_prices):
    assert [price_to_number(p) for p in raw_prices] == [160.71, 1083.0, 49.0]


@pytest.mark.parametrize(("text", "expected"), [("$0.50", 0.5), (" $85 ", 85.0), ("$10,025.00", 10025.0)])
def test_edge_cases(text, expected):
    assert price_to_number(text) == expected


@pytest.mark.parametrize("text", ["", "abc", "85 EUR", None, 85])
def test_malformed_price_raises(text):
    with pytest.raises(ValueError, match="expected a price"):
        price_to_number(text)


if __name__ == "__main__":
    raise SystemExit(pytest.main([__file__, "-q"]))   # 9 passed
```

In the workspace, tests live in `tests/` and import the package: `from listingtools import ListingRecord`. Run them with `uv run pytest -q` in the workspace folder.

### In practice

- NumPy, pandas and scikit-learn use pytest; their test suites run in continuous integration on every pull request before a maintainer merges it.
- In 2013, a reproduction by Herndon, Ash and Pollin found a spreadsheet formula error in an influential economics paper (Reinhart and Rogoff 2010) that had excluded several countries from an average. Automated checks of the computation against hand-computed cases are designed to catch exactly this kind of error.

> **Caution.** A test that never fails is worthless. After writing a test, break the code on purpose (or change the expected value) and check that the test fails.

> **Tip.** When you fix a bug, first write a test that reproduces it. The test fails, you fix the code, the test passes, and the bug can never come back unnoticed (a **regression test**).

## Worked example: a class that validates and cleans one listing

The [workspace](../workspace/README.md) contains `ListingRecord`, a dataclass that combines everything above. Its exercises 1 and 2 are part of the [exercise notebook](session-02-exercises.ipynb), part 1:

1. Read `src/listingtools/records.py`: which fields does `ListingRecord` have, which rules does `__post_init__` apply (price stored as text becomes a number, coordinates inside Berlin's bounding box, room type one of the four known values, `accommodates` at least 1), and what does `from_dict` add?
2. Run `uv run pytest -q tests/test_records.py` and read one parametrized test.
3. Complete the TODO: `maximum_nights` is optional, but if present it must not be smaller than `minimum_nights`; the raw file's sentinel 2,147,483,647 means "no limit". Activate the two tests of `maximum_nights`.
4. Add two tests of your own for rules that are not yet tested.

In [7]:
import sys

sys.path.insert(0, "sessions/02-software-engineering/workspace/src")   # uv run in the workspace does this
from listingtools import ListingRecord, ValidationError

raw = {"id": "3176", "name": " Fabulous  Flat ", "neighbourhood_group_cleansed": "Pankow",
       "latitude": "52.53574", "longitude": 13.41734, "room_type": "entire home/apt",
       "accommodates": 2, "price": "$1,160.50", "minimum_nights": "2"}
record = ListingRecord.from_dict(raw)
print(record.id, record.name, record.district, record.room_type)
# 3176 Fabulous Flat Pankow Entire home/apt
print(record.price, record.price_per_guest, record.is_short_stay)   # 1160.5 580.25 True
try:
    ListingRecord.from_dict({**raw, "latitude": 48.137})            # Munich
except ValidationError as err:
    print(err.field, "|", err)   # latitude | latitude: 48.137 lies outside Berlin (52.33 to 52.68)

3176 Fabulous Flat Pankow Entire home/apt
1160.5 580.25 True
latitude | latitude: 48.137 lies outside Berlin (52.33 to 52.68)


## Check your understanding

1. What is the difference between a function and a method, and between a class and an object? What is `self`?
2. `ListingTable` holds a pandas DataFrame as an attribute. Is this composition or inheritance, and why?
3. What does `@dataclass` generate, and what is `__post_init__` for?
4. A function has the hint `price: float`. What happens when it is called with the text `"$85.00"`?
5. Why does `ValidationError` inherit from both `ListingToolsError` and `ValueError`?
6. Name three kinds of cases a good set of tests for `parse_price` should contain.

## Further reading

- Downey, A. B. (2024). *Think Python*, 3rd edition, chapters 14–17 (classes and functions, classes and methods, classes and objects, inheritance). Free online, CC BY-NC-SA 4.0. https://allendowney.github.io/ThinkPython/
- The Carpentries Incubator (2024). *Intermediate Research Software Development in Python*, section 2 (testing) and section 3 (software design). CC-BY 4.0. https://carpentries-incubator.github.io/python-intermediate-development/
- Python Software Foundation (2026). *dataclasses — Data Classes*. https://docs.python.org/3/library/dataclasses.html
- pytest developers (2026). *Get started*; *How to parametrize fixtures and test functions*. https://docs.pytest.org/en/stable/getting-started.html